# LAB 2 — Medical Image Segmentation
**Medical Image Analysis · Biomedical Engineering · URJC 2026/2027**
**Group:** TODO · **Authors:** TODO

| Part | Content |
|---|---|
| **Part 1** | Thresholding, region growing and watershed · parameter influence · comparison and choice per ROI · pre-/post-processing |
| **Part 2** | Active contours (snakes) |
| **Part 3** | Additional state-of-the-art segmentation method |

In [ ]:
# --- Setup: libraries, project modules and reproducibility ---------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project root = folder that contains src/ (works whether the notebook runs from notebooks/ or the root)
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from src import utils, export_seg, preprocessing, thresholding, region_growing, watershed_seg

utils.ensure_dirs()          # creates results/figures, results/tables, ...
SEED = utils.set_seed(42)    # reproducibility

# PART 1 — Thresholding, Region Growing and Watershed

In [ ]:
# --- Step 2: load the two Lab 1 images and define the ROI of each one ------------------
# ROI = bounding box (ymin, ymax, xmin, xmax) in pixels -> img[ymin:ymax, xmin:xmax]
IMAGES = {
    "brain": {"file": "Brain_MRI_131749_T1.png",
              "structure": "Hypointense lesion (frontal white matter)",
              "roi": (120, 205, 125, 215)},
    "chest": {"file": "Chest_Xray_PA_3-8-2010.png",
              "structure": "Right lung",
              "roi": (140, 1910, 130, 1130)},
}

images, rois = {}, {}
for key, info in IMAGES.items():
    images[key] = utils.load_image(info["file"])          # 2-D grayscale image, float in [0, 1]
    rois[key] = utils.crop_roi(images[key], info["roi"])  # sub-image that will be segmented
    h, w, n_roi = utils.roi_dimensions(info["roi"])
    print(f"{key}: image {images[key].shape} -> ROI {info['roi']} = {h} x {w} px (N_ROI = {n_roi})")

    # Full image with the ROI in red vs. cropped ROI
    fig = utils.plot_roi_overview(images[key], info["roi"], title=info["structure"])
    utils.save_figure(fig, f"part1_roi_{key}.png")
    plt.show()

In [ ]:
# --- Step 3: manual reference segmentations (ground truth) made in 3D Slicer -----------
GT_FILES = {"brain": "SegmentationTumor.nrrd", "chest": "SegmentationLung.nrrd"}

gts = {}
for key, info in IMAGES.items():
    # Slicer may export only the box around the segment -> place it in the full image grid
    gt_full = export_seg.read_mask(utils.DATA_GT / GT_FILES[key], reference_shape=images[key].shape)
    gts[key] = utils.crop_roi(gt_full, info["roi"])   # same ROI as the image
    assert gts[key].sum() == gt_full.sum(), f"{key}: part of the reference lies outside the ROI"

    # Binary mask with the size of the full image (0 = background, 255 = structure)
    mask_path = export_seg.write_mask_png(gt_full, utils.DATA_GT / f"{key}_reference_mask.png")
    print(f"{key}: reference = {gt_full.sum()} px ({100 * gts[key].mean():.1f} % of the ROI) "
          f"-> {mask_path.relative_to(PROJECT_ROOT)}")

# Reference contour (green) over each ROI
fig, axes = plt.subplots(1, 2, figsize=(10, 6))
for ax, key in zip(axes, IMAGES):
    ax.imshow(rois[key], cmap="gray")
    ax.contour(gts[key], levels=[0.5], colors="lime")
    ax.set_title(f"Reference — {IMAGES[key]['structure']}")
    ax.axis("off")
utils.save_figure(fig, "part1_reference.png")
plt.show()

# PART 2 — Active Contours (Snakes)

# PART 3 — Other Segmentation Methods